# 3 · Inverted indexes and the SINTER tightening

<img src="https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120" alt="Redis"/>

<a href="https://colab.research.google.com/github/redis-field-engineering/redis-dsp-demo/blob/main/notebooks/03_sinter_tightening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part of the Redis DSP candidate-generation demo. The full sequence walks the bid path from naive to fast across six notebooks; this is `03_sinter_tightening.ipynb`.

**To run in Colab:** click the badge above, then *Runtime → Run all*. The setup cells below clone the repo, install dependencies, start a Redis Stack server, and load the synthetic dataset.

**To run locally:** make sure the docker-compose stack is up (`make up` from the repo root). The setup cells detect a local environment and skip the Colab-specific steps.

## Setup

These five cells prepare the environment. They are idempotent — safe to re-run, safe in either Colab or local. On Colab the first run takes about 60–90 seconds (pip install + apt install + dataset generation). Subsequent runs are near-instant because everything is cached.

In [1]:
# Setup 1/5 · clone the repo (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("pyproject.toml"):
    print("Cloning https://github.com/redis-field-engineering/redis-dsp-demo ...")
    os.system("git clone -q https://github.com/redis-field-engineering/redis-dsp-demo.git _repo")
    os.system("cp -R _repo/. ./")
    os.system("rm -rf _repo")
    print("Repo cloned.")
elif not IN_COLAB:
    print("Local environment detected — skipping clone.")
else:
    print("Repo already present.")

Local environment detected — skipping clone.


In [2]:
# Setup 2/5 · install Python dependencies (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        f'{sys.executable} -m pip install -q '
        '"redis[hiredis]>=5.2.0" "pydantic>=2.9.0" "pandas>=2.2.0" "pyarrow>=18.0.0"'
    )
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping pip install (assumes deps are already installed).")

Local environment detected — skipping pip install (assumes deps are already installed).


In [3]:
# Setup 3/5 · install and start Redis Stack (Colab only).
import os, sys, shutil
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("redis-stack-server") is None:
        print("Installing redis-stack-server ...")
        os.system(
            'curl -fsSL https://packages.redis.io/gpg | '
            'sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg'
        )
        os.system(
            'echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] '
            'https://packages.redis.io/deb $(lsb_release -cs) main" '
            '| sudo tee /etc/apt/sources.list.d/redis.list > /dev/null'
        )
        os.system("sudo apt-get update -qq > /dev/null 2>&1")
        os.system("sudo apt-get install -qq -y redis-stack-server > /dev/null 2>&1")
    os.system("redis-stack-server --daemonize yes > /dev/null 2>&1")
    print("redis-stack-server started on :6379")
else:
    print("Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).")

Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).


In [4]:
# Setup 4/5 · choose the Redis URL.
import os, sys
IN_COLAB = "google.colab" in sys.modules
default_port = "6379" if IN_COLAB else "6381"
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", default_port)
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
auth = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
REDIS_URL = f"redis://{auth}{REDIS_HOST}:{REDIS_PORT}/0"
os.environ["DEMO_REDIS_URL"] = REDIS_URL
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://localhost:6381/0


In [5]:
# Setup 5/5 · generate and load the synthetic dataset (only if Redis is empty).
import sys, subprocess
from pathlib import Path

# Find the repo root so we can run `python -m data.synthetic` reliably.
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from redis import Redis
client = Redis.from_url(REDIS_URL, decode_responses=True)
if not client.ping():
    raise RuntimeError(f"Redis at {REDIS_URL} did not answer PING")

if client.exists("meta:dataset_loaded"):
    print(
        f"Dataset already loaded: "
        f"{client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )
else:
    print("Generating synthetic dataset (~30 seconds) ...")
    subprocess.run(
        [sys.executable, "-m", "data.synthetic",
         "--output", "data/generated/synthetic",
         "--num-users", "4000",
         "--num-campaigns", "2500",
         "--num-interactions", "120000",
         "--feature-count", "12"],
        cwd=_repo_root, check=True,
    )
    print("Loading dataset into Redis ...")
    subprocess.run(
        [sys.executable, "-m", "data.load_redis",
         "--redis-url", REDIS_URL,
         "--dataset-dir", "data/generated/synthetic"],
        cwd=_repo_root, check=True,
    )
    print(
        f"Done. {client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )

Dataset already loaded: 4000 users, 2500 campaigns.


## Walkthrough

From here on the notebook is the demonstration.

In [6]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What an inverted index looks like

`idx:geo:<country>` is a Redis SET containing the campaign IDs whose
targeting includes that country. Same for `idx:state:`, `idx:device:`,
`idx:device_type:`, `idx:card_tier:`, and `idx:segment:`.


In [7]:
for key in ['idx:geo:US', 'idx:device:iOS', 'idx:card_tier:Gold',
            'idx:segment:travel_high', 'idx:segment:gaming_high']:
    members = client.smembers(key)
    print(f'  {key:<28} {len(members):>5} campaigns; sample: {sorted(members)[:3]}')

  idx:geo:US                    1216 campaigns; sample: ['c00000', 'c00003', 'c00006']
  idx:device:iOS                1564 campaigns; sample: ['c00000', 'c00003', 'c00005']
  idx:card_tier:Gold            1491 campaigns; sample: ['c00000', 'c00001', 'c00003']
  idx:segment:travel_high        226 campaigns; sample: ['c00012', 'c00033', 'c00036']
  idx:segment:gaming_high        244 campaigns; sample: ['c00003', 'c00006', 'c00033']


## A sample MAID

Use the same MAID across all three notebooks (3 → 4 → 5) so the comparison
is apples-to-apples.


In [8]:
from app.models import UserProfile
profile = client.hgetall('maid:maid_00042')
user = UserProfile.from_redis_hash(profile)
print(f'maid_id     = {user.user_id}')
print(f'geo / state = {user.geo} / {user.state}')
print(f'device      = {user.device_type} / {user.device}')
print(f'card_tier   = {user.card_tier}')
print(f'segments    = {user.segments[:6]}')

maid_id     = maid_00042
geo / state = US / IL
device      = ctv / Roku
card_tier   = Standard
segments    = ['fitness_high', 'finance_medium', 'gaming_medium', 'streaming_medium', 'tech_medium']


## The bruteforce 26-probe plan

The legacy planner explores combinations of geo, state, device, device_type,
card_tier, and the user's strong segments. Each probe is a separate
`SINTER` call and each call is a separate Redis round trip.


In [9]:
from app.candidate import build_legacy_union_probe_candidate_lookup_keys
legacy_probes = build_legacy_union_probe_candidate_lookup_keys(user, strong_signal_count=2)
print(f'probe count = {len(legacy_probes)}')
print()
print('first 6 probes:')
for keys in legacy_probes[:6]:
    print(f'  SINTER ' + ' '.join(keys))
print('  ...')
print(f'last probe:')
print(f'  SINTER ' + ' '.join(legacy_probes[-1]))

probe count = 26

first 6 probes:
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:device:Roku idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device:Roku idx:segment:finance_medium
  ...
last probe:
  SINTER idx:geo:US


Run the bruteforce plan one round trip at a time, the way the
legacy code path does it.


In [10]:
timer = StepTimer()

with timer.step('legacy_26_probes_sequential'):
    probe_results = []
    for keys in legacy_probes:
        result = client.sinter(keys)   # one Redis round trip per probe
        if result:
            probe_results.append(sorted(result))

unique_candidates = set()
for batch in probe_results:
    unique_candidates.update(batch)

print(f'unique campaign IDs returned: {len(unique_candidates)}')
print(f'round trips:                  {len(legacy_probes)}')
print()
print(timer.summary())

unique campaign IDs returned: 1453
round trips:                  26

     legacy_26_probes_sequential    9.388 ms
--------------------------------------------
                           TOTAL    9.388 ms


## The tightened 3-probe plan

The tightened planner notices that 26 probes were not actually buying us
much recall. The compact plan keeps:

- one probe per strong segment (with the strict geo + device + card_tier base),
- plus one strict-base fallback probe.

That's typically `3` probes for a 2-strong-segment user. And — the bigger
win — they are issued in a single pipeline, so it costs **one** Redis
round trip instead of N.


In [11]:
from app.candidate import build_union_probe_candidate_lookup_keys
tight_probes = build_union_probe_candidate_lookup_keys(user, strong_signal_count=2)
print(f'probe count = {len(tight_probes)}')
print()
for keys in tight_probes:
    print(f'  SINTER ' + ' '.join(keys))

probe count = 3

  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku


In [12]:
with timer.step('tightened_3_probes_pipelined'):
    pipe = client.pipeline(transaction=False)
    for keys in tight_probes:
        pipe.sinter(keys)
    pipelined_results = pipe.execute()       # one round trip total

unique_tightened = set()
for batch in pipelined_results:
    unique_tightened.update(batch)

print(f'unique campaign IDs returned: {len(unique_tightened)}')
print(f'round trips:                  1   (pipelined)')
print()
print(timer.summary())

unique campaign IDs returned: 190
round trips:                  1   (pipelined)

     legacy_26_probes_sequential    9.388 ms
    tightened_3_probes_pipelined    0.608 ms
--------------------------------------------
                           TOTAL    9.996 ms


## What the timing tells us

Two effects compound here:

1. The probe count drops from ~26 to 3, so Redis does less SET algebra.
2. The round-trip count drops from ~26 to 1 because the tightened plan
   pipelines all three SINTER calls in a single batch.

In a tuned-VM run, the bruteforce mode lands at `~18 ms` p50 decision-path
and the tightened mode lands at `~4.3 ms`. Most of that gap is the
sequential round-trip cost on the bruteforce side.

The candidate set is still ~50 ads at this stage — narrower than the 2500
of `full_realtime`, but the bid path now has to fetch each candidate's
metadata, evaluate the full per-campaign rules, and rerank. Notebook 4
removes most of *that* cost by doing the candidate generation offline.
